# Train YOLOv8 Weed Detector

This notebook guides you through training a custom YOLOv8 model to detect weeds using the [CropWeeds-YOLO Dataset](https://www.kaggle.com/datasets/swish9/weeds-detection).

The resulting model can be used in the `LawnAnalyzer` class to improve weed detection accuracy.

## 1. Install Dependencies

Ensure `ultralytics` is installed. We also need `kaggle` to download the dataset.

In [2]:
%pip install ultralytics kaggle

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 2. Download Dataset from Kaggle

**Prerequisite:** You need a `kaggle.json` API token.
1. Go to your Kaggle Account settings.
2. Click "Create New API Token".
3. Place the `kaggle.json` file in `C:\Users\<YourUser>\.kaggle\` (Windows) or `~/.kaggle/` (Linux/Mac).

Alternatively, you can manually download the dataset from [here](https://www.kaggle.com/datasets/swish9/weeds-detection), unzip it, and place it in a folder named `datasets/weeds-detection` in the project root.

The code below attempts to download it automatically.

In [3]:
import os
from pathlib import Path
 
# Define paths
project_root = Path("..").resolve() # Assuming this notebook is in models/
dataset_dir = project_root / "datasets" / "weeds-detection"

# Create datasets directory if it doesn't exist
dataset_dir.mkdir(parents=True, exist_ok=True)

# Check if dataset already exists
if (dataset_dir / "images").exists() and (dataset_dir / "labels").exists():
    print(f"Dataset found at {dataset_dir}")
else:
    # Download dataset using Kaggle API
    # Note: This requires kaggle.json to be set up correctly
    try:
        import kaggle
        print("Downloading dataset...")
        kaggle.api.dataset_download_files('swish9/weeds-detection', path=dataset_dir, unzip=True)
        print("Download complete.")
    except Exception as e:
        print(f"Could not download automatically: {e}")
        print(f"Please manually download the dataset to: {dataset_dir}")

Dataset found at G:\DSBA\DSBA 6211\Project\Gardening-Application\datasets\weeds-detection


## 3. Prepare Dataset Structure

The dataset should already be in YOLO format. Let's verify the structure.
It typically contains `train`, `test`, and `val` folders, each with `images` and `labels`.

In [4]:
# Verify structure
print(f"Checking contents of {dataset_dir}...")
for item in dataset_dir.iterdir():
    print(item.name)

# Define paths for data.yaml
train_path = dataset_dir / "images" / "train"
val_path = dataset_dir / "images" / "val"

if not train_path.exists():
    print(f"Warning: Train path {train_path} does not exist. Check the unzipped structure.")
else:
    print(f"Train path verified: {train_path}")

Checking contents of G:\DSBA\DSBA 6211\Project\Gardening-Application\datasets\weeds-detection...
classes.txt
data.yaml
images
labels
Train path verified: G:\DSBA\DSBA 6211\Project\Gardening-Application\datasets\weeds-detection\images\train


## 4. Create YOLO Configuration (YAML)

We need to create a `data.yaml` file that tells YOLO where the images are and what the classes are.
Based on the dataset description, it detects weeds and crops. We need to check the `data.yaml` if it came with one, or create our own.

In [5]:
import yaml

# Define the data configuration
# The dataset structure is:
# datasets/weeds-detection/
#   images/
#     train/
#     val/
#     test/
#   labels/
#     train/
#     val/
#     test/

data_config = {
    'path': str(dataset_dir.absolute()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 2,
    'names': ['crop', 'weed'] 
}

yaml_path = dataset_dir / "data.yaml"

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print(f"Created configuration at {yaml_path}")

Created configuration at G:\DSBA\DSBA 6211\Project\Gardening-Application\datasets\weeds-detection\data.yaml


## 5. Initialize YOLOv8 Model

We will use `yolov8n.pt` (Nano) as the starting point. It's small and fast, suitable for the application.

In [6]:
from ultralytics import YOLO

# Load a model
model = YOLO('yolov8n.pt')  # load a pretrained model (recommended for training)

## 6. Train the Model

Train the model for a specified number of epochs. 50-100 is usually a good start.

In [ ]:
import torch

# Check for GPU availability
if torch.cuda.is_available():
    device = 0 # Use the first GPU
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    device = 'cpu'
    print("No GPU detected. Training on CPU.")

# Train the model
# Optimizations for speed:
# - Reduced epochs to 25 (from 50)
# - Reduced image size to 416 (from 640) - faster processing
# - Reduced patience to 5 - stops sooner if not improving
# - Added cache=True to keep images in RAM
# - Explicitly set device
results = model.train(
    data=str(yaml_path),
    epochs=25,
    imgsz=416,
    patience=5,
    batch=16,
    cache=True,
    device=device,
    name='yolov8n_weeds'
)

Ultralytics 8.3.235  Python-3.13.9 torch-2.9.1+cpu CPU (Intel Core i7-14700KF)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=G:\DSBA\DSBA 6211\Project\Gardening-Application\datasets\weeds-detection\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_weeds10, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_ma

## 7. Evaluate Model Performance

Check the validation metrics.

In [ ]:
# Validate the model
metrics = model.val()
print(f"mAP50-95: {metrics.box.map}")

## 8. Export Model

Save the best model to the `models/` directory so the app can use it.

In [ ]:
import shutil

# The best model is saved in runs/detect/yolov8n_weeds/weights/best.pt
best_model_path = Path(results.save_dir) / 'weights' / 'best.pt'
target_path = project_root / 'models' / 'weed_detector.pt'

if best_model_path.exists():
    shutil.copy(best_model_path, target_path)
    print(f"Model saved to {target_path}")
    print("Update your LawnAnalyzer class to load this model: self.yolo_model = YOLO('models/weed_detector.pt')")
else:
    print("Could not find best.pt")